# Text chunking strategies

RAG step one: cut the document into chunks, so that only the relevant ones go into the prompt.
Bad chunks bring the wrong context: a question about software bugs gets a chunk about a medical
study, or a header lands in one chunk and its text in the next.

| Strategy | How | Good | Bad |
|---|---|---|---|
| **Size** | every N characters, with overlap | works on anything, code included | cuts words and separates headers from their text |
| **Structure** | at the document's headers (`## `) | the cleanest chunks | needs reliable markup |
| **Sentence** | N sentences, with sentence overlap | a middle ground for plain text | misses structure without punctuation |
| **Semantic** | group sentences by how related they are (NLP) | the most relevant chunks | slow and complex; no code in the course |

Choosing: structure when you control the format, sentences for most text,
size with overlap as the production fallback. There is no best strategy for every case.

The functions live in [`chunking.py`](./chunking.py) for the next RAG lessons. The document is the course's
[`report.md`](./report.md): an annual report of a made-up company, one `## ` section per department.

## Setup

In [1]:
from pathlib import Path

from chunking import chunk_by_char, chunk_by_section, chunk_by_sentence

text = Path("report.md").read_text()
print(f"{len(text)} characters, a title and {text.count(chr(10) + '## ')} sections")

18305 characters, a title and 14 sections


In [2]:
def stats(chunks):
    lengths = [len(chunk) for chunk in chunks]
    # a "## " header anywhere but at the very start = the chunk mixes two sections
    mixed = sum(1 for chunk in chunks if "## " in chunk[1:])
    return f"{len(chunks):3} chunks, {min(lengths):5}-{max(lengths):5} chars, avg {sum(lengths) // len(chunks):5}, mixed sections: {mixed}"

## By size

150 characters with 20 of overlap, the course defaults. Look at the chunks around the start of
section 2: words are cut, and the header sits at the end of a medical chunk.

In [3]:
chunks = chunk_by_char(text, chunk_size=150, chunk_overlap=20)
print(stats(chunks))

# chunk i starts at i * (150 - 20): find the one that holds the section 2 header
start = text.find("## Section 2") // (150 - 20)
for i in range(start - 1, start + 2):
    print(f"\n[{i}] {chunks[i]!r}")

141 chunks,   105-  150 chars, avg   149, mixed sections: 17

[39] 'lysis) may impact the pace of subsequent trial phases. The team continues to refine diagnostic protocols based on this evolving understanding.\n\n## Sec'

[40] 'derstanding.\n\n## Section 2: Software Engineering - Project Phoenix Stability Enhancements\n\nThe Software Engineering division dedicated considerable ef'

[41] 'ated considerable effort to improving the stability and performance of the core systems underpinning Project Phoenix. Recurring issues, particularly `'


Bigger chunks are closer to production sizes: fewer cut points, but each chunk still mixes sections.

In [4]:
print(stats(chunk_by_char(text, chunk_size=1000, chunk_overlap=200)))

 23 chunks,   705- 1000 chars, avg   987, mixed sections: 19


Overlap must be smaller than the chunk, otherwise the course loop never moves forward. Here it raises instead:

In [5]:
try:
    chunk_by_char(text, chunk_size=100, chunk_overlap=100)
except ValueError as e:
    print(e)

chunk_overlap must be at least 0 and less than chunk_size


## By sentence

5 sentences with 1 of overlap. Whole sentences, but a header has no period at the end,
so it sticks to the next sentence, and a chunk can still start in one section and end in another.

In [6]:
chunks = chunk_by_sentence(text, max_sentences_per_chunk=5, overlap_sentences=1)
print(stats(chunks))

i = next(i for i, chunk in enumerate(chunks) if "## Section 2" in chunk)
print(f"\n[{i}] {chunks[i][:700]}...")

 33 chunks,   237- 1023 chars, avg   701, mixed sections: 15

[10] These results provide a much-needed foundation for future clinical strategies, though the resource implications highlighted in Section 3 (Financial Analysis) may impact the pace of subsequent trial phases. The team continues to refine diagnostic protocols based on this evolving understanding. ## Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems underpinning Project Phoenix. Recurring issues, particularly `ERR_MEM_ALLOC_FAIL_0x8007000E` during peak loads and `TIMEOUT_QUERY_DB_0xDEADBEEF` affecting data retrieval operations, were prioritized, at...


## By section

One chunk per `## ` section. The course splits on `\n## ` and so drops the `## ` from every chunk;
here the split happens just before the header, and the header stays with its text.

In [7]:
chunks = chunk_by_section(text)
print(stats(chunks), "\n")
for chunk in chunks:
    print(f"{len(chunk):5}  {chunk.splitlines()[0][:80]}")

 15 chunks,    69- 2187 chars, avg  1218, mixed sections: 0 

   69  # **Annual Interdisciplinary Research Review: Cross-Domain Insights**
 1962  ## Executive Summary
  856  ## Table of Contents
 1137  ## Methodology
 1180  ## Section 1: Medical Research - Understanding XDR-471 Syndrome
 1245  ## Section 2: Software Engineering - Project Phoenix Stability Enhancements
 1172  ## Section 3: Financial Analysis - Q3 Performance and Outlook
 1183  ## Section 4: Scientific Experimentation - Characterization of Material Composit
 1225  ## Section 5: Legal Developments - Navigating IP Precedents and Regulatory Shift
 1144  ## Section 6: Product Engineering - Finalizing Model Zircon-5 Specifications
 1232  ## Section 7: Historical Research - Re-evaluating the Galveston Accords (1921)
 1175  ## Section 8: Project Management - Progress on Project Cerberus Phase 2B
 1159  ## Section 9: Pharmaceutical Development - Compound CTX-204b Phase IIa Update
 1350  ## Section 10: Cybersecurity Analysis - In

In [8]:
print(next(chunk for chunk in chunks if chunk.startswith("## Section 2"))[:500], "...")

## Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems underpinning Project Phoenix. Recurring issues, particularly `ERR_MEM_ALLOC_FAIL_0x8007000E` during peak loads and `TIMEOUT_QUERY_DB_0xDEADBEEF` affecting data retrieval operations, were prioritized, at a cost of INC-2023-Q4-011. Root cause analysis pointed towards inefficiencies in the primary  ...


## Side by side

"Mixed sections" counts the chunks that hold the end of one section and the start of another:
exactly what brings a medical paragraph into an answer about software.

In [9]:
for name, chunks in [
    ("size 150/20", chunk_by_char(text, 150, 20)),
    ("size 1000/200", chunk_by_char(text, 1000, 200)),
    ("sentence 5/1", chunk_by_sentence(text, 5, 1)),
    ("section", chunk_by_section(text)),
]:
    print(f"{name:14} {stats(chunks)}")

size 150/20    141 chunks,   105-  150 chars, avg   149, mixed sections: 17
size 1000/200   23 chunks,   705- 1000 chars, avg   987, mixed sections: 19
sentence 5/1    33 chunks,   237- 1023 chars, avg   701, mixed sections: 15
section         15 chunks,    69- 2187 chars, avg  1218, mixed sections: 0


The section chunks are clean, but uneven in size: the title alone is a 69-character chunk,
"Future Directions" is over 2000, and a long section could be too big for one embedding. Real pipelines often combine strategies: split by section,
then by sentence or size inside the long sections.

Next lesson: text embeddings.